# LC 347 — Top K Frequent Elements
**Difficulty:** Medium &nbsp;|&nbsp;
**Category:** HashMap + Heap / Bucket Sort
**Pattern:** Count Then Rank — Frequency Map + Top K

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Count frequencies with a
HashMap, then find the top k by frequency. Bucket
sort is O(n) because frequency can never exceed
len(nums) — use that as the array index.
</div>

## Official Problem Statement

Given an integer array `nums` and an integer `k`,
return the `k` most frequent elements. You may
return the answer in any order.

**Example 1:**
```
Input:  nums = [1,1,1,2,2,3], k = 2
Output: [1,2]
```
**Example 2:**
```
Input:  nums = [1], k = 1
Output: [1]
```

**Constraints:**
- `1 <= nums.length <= 10^5`
- `-10^4 <= nums[i] <= 10^4`
- `k` is in the range `[1, number of unique elements]`
- It is guaranteed the answer is unique

## What This Is Actually Asking

Count how many times each number appears in the list.
Then return the k numbers with the highest counts.
Order of the returned numbers does not matter.
The answer is always unique — no ties at the k-th
position.

## Walk Through an Example by Hand

```
nums = [1, 1, 1, 2, 2, 3]   k = 2

Step 1 — Count frequencies:
  count = {1:3, 2:2, 3:1}

Step 2 — Bucket sort (index = frequency):
  buckets = [[], [], [], [], [], [], []]  (len 7 = n+1)
  index:      0   1   2   3   4   5   6

  num=1 freq=3 -> buckets[3] = [1]
  num=2 freq=2 -> buckets[2] = [2]
  num=3 freq=1 -> buckets[1] = [3]

  buckets = [[], [3], [2], [1], [], [], []]

Step 3 — Collect from highest bucket down:
  i=6: empty
  i=5: empty
  i=4: empty
  i=3: [1]  result=[1]  len=1 < k=2, keep going
  i=2: [2]  result=[1,2] len=2 == k=2 -> done

Answer: [1, 2]
```

## The Picture

```
nums = [1,1,1,2,2,3]   k=2

Frequency map:
  1 -> 3  |###|
  2 -> 2  |## |
  3 -> 1  |#  |

Bucket sort — use frequency AS the index:

  bucket[6] []
  bucket[5] []
  bucket[4] []
  bucket[3] [1]   <- most frequent
  bucket[2] [2]
  bucket[1] [3]   <- least frequent
  bucket[0] []

Walk from the top: grab until you have k elements.
Frequency can never exceed n — that caps the bucket
array size and makes this O(n).

Heap alternative (O(n log k)):
  heapq.nlargest(k, count.keys(), key=count.get)
  Good for large n, small k.
```

## When To Use This Pattern

- When you see **"top k by frequency"**, think
  **count with Counter, then rank**
- When frequency is bounded by n, think **bucket
  sort — O(n) with frequency as array index**
- When n is very large and k is small, think
  **min-heap of size k — O(n log k)**
- When you want the single most frequent element
  (k=1), think **Counter.most_common(1)**

## The Approach

Count the frequency of every element using a
dictionary, then build a bucket array of length
n+1 where the index represents the frequency.
Place each element into the bucket matching its
frequency count.
Walk the bucket array from the highest index downward,
collecting elements until k elements have been
gathered, then return the result.

In [1]:
from typing import List           # type hints
from collections import Counter   # fast frequency count

In [20]:
def test_harness(func):
    tests = [
        # (nums, k, expected — order-insensitive)
        ([1,1,1,2,2,3],    2, [1,2]),
        ([1],              1, [1]),
        ([1,2],            2, [1,2]),
        ([4,1,2,2,3,3,3],  2, [3,2]),
        ([1,1,1,2,2,3],      1, [1]),   # tie: either ok
        ([-1,-1,2,2,2],    1, [2]),
        ([5,5,5,5,4,4,3,2,1], 3, [5,4,3]),
    ]

    passed = 0
    for i, (nums, k, expected) in enumerate(tests):
        result = func(nums[:], k)
        # order doesn't matter — compare as sets
        ok = set(result) == set(expected) and len(result)==k
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"nums={nums} k={k} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [ ]:
def topKFrequent(nums: List[int], k: int) -> List[int]:
    """
    Return the k most frequent elements in nums.

    Count frequencies with a dict. Build a bucket array
    of size n+1 (index = frequency). Place each number
    in its bucket. Walk buckets from highest to lowest,
    collecting until k elements are gathered.

    Time:  O(n) — count + fill + collect all O(n)
    Space: O(n) — count map + bucket array
    """
    pass


# Quick debug — run this cell while building
print(topKFrequent([1,1,1,2,2,3], 2))    # [1,2]
print(topKFrequent([1], 1))               # [1]
print(topKFrequent([-1,-1,2,2,2], 1))    # [2]
print(topKFrequent([4,1,2,2,3,3,3], 2))  # [3,2]

In [ ]:
# Uncomment and run when solution is ready
# test_harness(topKFrequent)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Sort by frequency | O(n log n) | O(n) |
| Min-heap of size k | O(n log k) | O(n+k) |
| Bucket sort | O(n) | O(n) |

Bucket sort wins on time because frequency is
naturally bounded by n — no comparison sort needed.

## Real World Connection

At Citi, identifying the top-k error codes from
millions of telemetry log lines is a daily ops task
— which errors are responsible for most alerts?
A Counter over the error_code column followed by
bucket sort gives the answer in O(n), fast enough
to run inline in the Lambda ingestion function
before flushing to S3.
The heap variant (O(n log k)) is the right choice
in Glue/PySpark where k is small (top 10 servers)
but n is billions of rows across the data lake —
it avoids materialising the full frequency map.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra

In [21]:
from collections import Counter
def topKFrequent(nums: List[int], k: int) -> List[int]:
    """
    Return the k most frequent elements in nums.

    Count frequencies with a dict. Build a bucket array
    of size n+1 (index = frequency). Place each number
    in its bucket. Walk buckets from highest to lowest,
    collecting until k elements are gathered.

    Time:  O(n) — count + fill + collect all O(n)
    Space: O(n) — count map + bucket array
    """
    # create a bucket for every possible numbers to come out in the output list.
    #bukets will contain the indexes of the item
    # 1. Count frequencies in  counts
    counts = Counter(nums)
    # 2. Build bucket array (index = frequency)
    # The size is len(nums) + 1 because the max frequency is n
    buckets = [[] for _ in range(len(nums) + 1)]
    for num, freq in counts.items():
        buckets[freq].append(num)
# 3. Iterate backwards from highest frequency to collect k items
    res = []
    for i in range(len(buckets) - 1, 0, -1):
        for n in buckets[i]:
            res.append(n)
            if len(res) == k:
                return res
    return res
        
# Quick debug — run this cell while building
print(topKFrequent([1,1,1,2,2,3], 2))    # [1,2]
print(topKFrequent([1], 1))               # [1]
print(topKFrequent([-1,-1,2,2,2], 1))    # [2]
print(topKFrequent([4,1,2,2,3,3,3], 2))  # [3,2]
test_harness(topKFrequent)

[1, 2]
[1]
[2]
[3, 2]
Test 1: PASSED | nums=[1, 1, 1, 2, 2, 3] k=2 | expected=[1, 2] | got=[1, 2]
Test 2: PASSED | nums=[1] k=1 | expected=[1] | got=[1]
Test 3: PASSED | nums=[1, 2] k=2 | expected=[1, 2] | got=[1, 2]
Test 4: PASSED | nums=[4, 1, 2, 2, 3, 3, 3] k=2 | expected=[3, 2] | got=[3, 2]
Test 5: PASSED | nums=[1, 1, 1, 2, 2, 3] k=1 | expected=[1] | got=[1]
Test 6: PASSED | nums=[-1, -1, 2, 2, 2] k=1 | expected=[2] | got=[2]
Test 7: PASSED | nums=[5, 5, 5, 5, 4, 4, 3, 2, 1] k=3 | expected=[5, 4, 3] | got=[5, 4, 3]

7/7 tests passed


In [22]:
import heapq
from collections import Counter
r"""
For Demo only and learning 
Min-Heap Implementation ($O(n \log k)$)While your Bucket Sort was $O(n)$, 
the Min-Heap approach is the standard 
alternative when memory is tighter or you're dealing with a data stream.
"""
def topKFrequent(nums: list[int], k: int) -> list[int]:
    # 1. Count frequencies - O(n)
    counts = Counter(nums)
    
    # 2. Maintain a Min-Heap of size k - O(n log k)
    heap = []
    
    for num, freq in counts.items():
        # Push (frequency, number) tuple. 
        # heapq sorts by the first element of the tuple.
        heapq.heappush(heap, (freq, num))
        
        # If heap exceeds size k, pop the element with the SMALLEST frequency
        if len(heap) > k:
            heapq.heappop(heap)
            
    # 3. Extract the numbers from the heap - O(k log k)
    return [num for freq, num in heap]

# Quick debug
print(topKFrequent([1,1,1,2,2,3], 2))    # [2, 1] (order may vary)
print(topKFrequent([4,1,2,2,3,3,3], 2))  # [2, 3]
# Quick debug — run this cell while building
print(topKFrequent([1,1,1,2,2,3], 2))    # [1,2]
print(topKFrequent([1], 1))               # [1]
print(topKFrequent([-1,-1,2,2,2], 1))    # [2]
print(topKFrequent([4,1,2,2,3,3,3], 2))  # [3,2]
test_harness(topKFrequent)
pass

[2, 1]
[2, 3]
[2, 1]
[1]
[2]
[2, 3]
Test 1: PASSED | nums=[1, 1, 1, 2, 2, 3] k=2 | expected=[1, 2] | got=[2, 1]
Test 2: PASSED | nums=[1] k=1 | expected=[1] | got=[1]
Test 3: PASSED | nums=[1, 2] k=2 | expected=[1, 2] | got=[1, 2]
Test 4: PASSED | nums=[4, 1, 2, 2, 3, 3, 3] k=2 | expected=[3, 2] | got=[2, 3]
Test 5: PASSED | nums=[1, 1, 1, 2, 2, 3] k=1 | expected=[1] | got=[1]
Test 6: PASSED | nums=[-1, -1, 2, 2, 2] k=1 | expected=[2] | got=[2]
Test 7: PASSED | nums=[5, 5, 5, 5, 4, 4, 3, 2, 1] k=3 | expected=[5, 4, 3] | got=[3, 5, 4]

7/7 tests passed
